# GraphCastSmall — Myanmar 24h Precipitation Forecast

**Model**: `earth2studio.models.px.GraphCastSmall` (DeepMind/Google, via NVIDIA Earth2Studio)  
**Resolution**: 1.0° global (181 × 360)  
**Timestep**: 6h native — no interpolation between steps  
**Horizon**: 24h (4 AR steps → t+6h, t+12h, t+18h, t+24h)  
**Variable**: `tp06` — 6-hour accumulated precipitation  
**Transform**: physical metres × 1000 → mm/6h — **no log/exp transform**  
**Init source**: ARCO ERA5 (historical) or IFS HRES (near-real-time)

---

## Before you start

1. **Select a GPU runtime**: Runtime → Change runtime type → T4 GPU
2. **Add your GitHub token** (for Section 14 push): Edit → Notebook settings → Secrets → Add `GITHUB_TOKEN`
   - Fine-grained PAT: GitHub → Settings → Developer Settings → Personal Access Tokens
   - Grant **Contents: Read and Write** on `JiayanLim/myanmar-weather-forecast`
3. Run cells **in order**. The smoke test (Section 7) must pass before the full forecast.

---

## Hardware gate (Constitution §XI)

GraphCastSmall recommended VRAM badge: **40 GB**. This notebook targets **T4 (16 GB)**.
T4 compatibility is **unverified** until this notebook completes successfully.

Staged test:

| Stage | Test | On OOM |
|-------|------|--------|
| 1 | GPU baseline (before model load) | N/A |
| 2 | Model load + VRAM snapshot | Stop |
| 3 | nsteps=1 smoke test | **Stop — report failure** |
| 4 | nsteps=4 full 24h run | Stop |

**If Stage 3 OOMs: do not attempt workarounds. Record the failure and stop.**

---

## Output

```
data/forecast/
  precipitation.bin   [5 × 21 × 11] float32  (t+0h..t+24h, 6h steps)  ~4.6 KB
  forecast.json       schema v2.0, is_demo=false
```

## Section 1 — Install dependencies

In [ ]:
import sys, os, subprocess

# earth2studio caches failed imports at module load time.
# If chex is not importable BEFORE earth2studio is first imported,
# the failure is stored permanently for the session.
# → We must install AND restart the kernel before any earth2studio import.

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

try:
    import chex, haiku, graphcast  # already installed + kernel already restarted
    print(f"chex {chex.__version__} ✓  |  haiku {haiku.__version__} ✓  |  graphcast ✓")
    print("All GraphCast deps present — continue to Section 2.")

except ImportError:
    print("Installing GraphCast dependencies...")
    _pip("chex", "dm-haiku")
    _pip("graphcast @ git+https://github.com/google-deepmind/graphcast.git")
    _pip("earth2studio[data]>=0.17.0", "numpy", "xarray", "zarr")
    print("\nInstall complete. Restarting runtime to clear import caches...")
    print("After restart: Runtime → Run all (or re-run from Section 1 to confirm ✓).")
    os.kill(os.getpid(), 9)  # SIGKILL → Colab auto-restarts kernel

## Section 2 — Clone repository

In [ ]:
import os, sys, subprocess

REPO_SLUG = "JiayanLim/myanmar-weather-forecast"
REPO_DIR  = "/content/myanmar-weather-forecast"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", f"https://github.com/{REPO_SLUG}.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--rebase"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

## Section 3 — GPU environment verification

Run **before** any model imports for a clean VRAM baseline.
Expected: NVIDIA T4, ~16 GB VRAM, CUDA 12.x

In [ ]:
import subprocess, torch, jax, earth2studio

print("=" * 60)
print("GPU ENVIRONMENT")
print("=" * 60)

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free,memory.used",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if smi.returncode == 0:
    p = [x.strip() for x in smi.stdout.strip().split(",")]
    print(f"  GPU model      : {p[0]}")
    print(f"  Driver version : {p[1]}")
    print(f"  VRAM total     : {p[2]}")
    print(f"  VRAM free      : {p[3]}")
    print(f"  VRAM used      : {p[4]}")
else:
    print("  WARNING: nvidia-smi not available")

nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
for line in nvcc.stdout.splitlines():
    if "release" in line.lower():
        print(f"  CUDA version   : {line.strip()}")

print(f"  PyTorch        : {torch.__version__}")
print(f"  JAX            : {jax.__version__}")
print(f"  Earth2Studio   : {earth2studio.__version__}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    VRAM_TOTAL_GB = props.total_memory / 1e9
    GPU_NAME = props.name
    print(f"  VRAM (torch)   : {VRAM_TOTAL_GB:.2f} GB total, {torch.cuda.mem_get_info()[0]/1e9:.2f} GB free")
    if VRAM_TOTAL_GB < 16.0:
        print(f"  ⚠  < 16 GB — GraphCastSmall (40 GB badge) very likely to OOM")
    elif VRAM_TOTAL_GB < 40.0:
        print(f"  ℹ  {VRAM_TOTAL_GB:.0f} GB < 40 GB rec badge — running staged test")
else:
    raise RuntimeError("No CUDA GPU. Select Runtime → Change runtime type → T4 GPU")

print("=" * 60)

## Section 4 — JAX memory configuration

**Must be set before any JAX model operations.**

- `XLA_PYTHON_CLIENT_PREALLOCATE=false` — prevents JAX from reserving ~90% of VRAM at import
- `XLA_PYTHON_CLIENT_MEM_FRACTION=0.85` — caps JAX allocation to 85% when it does allocate
- Note: JAX manages its own GPU memory pool, separate from PyTorch's allocator. `torch.cuda.memory_allocated()` does **not** reflect JAX model weights.

In [ ]:
import os

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print(f"XLA_PYTHON_CLIENT_PREALLOCATE : {os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']}")
print(f"XLA_PYTHON_CLIENT_MEM_FRACTION: {os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION']}")

# Pipeline constants — must match generate_forecast.py
MYANMAR_LAT_MIN = 9.0
MYANMAR_LAT_MAX = 29.0
MYANMAR_LON_MIN = 92.0
MYANMAR_LON_MAX = 102.0
GC_STEP_HOURS   = 6
GC_N_STEPS      = 4    # 4 AR steps → 24h horizon
GC_N_FRAMES     = 5    # prepend t+0h zero frame → 5 total
GC_N_LAT        = 21   # 9°N–29°N at 1.0°
GC_N_LON        = 11   # 92°E–102°E at 1.0°
SANITY_MAX_MM   = 500.0

print("Constants: Myanmar 21 lat × 11 lon at 1.0°, 5 frames, 6h steps")

## Section 5 — Data source setup

Initializes the initialization data source.

- **ARCO** (default): ERA5 reanalysis on Google Cloud, 1959–2023, free, no credentials. Use for historical init dates.
- **IFS**: ECMWF open data, near-real-time, free, no credentials. Use for current forecasts.

Set `SOURCE = "ifs"` and `INIT_TIME_STR = None` for a live operational forecast.

In [ ]:
from datetime import datetime, timedelta, timezone

# ── Choose source and init time ───────────────────────────────────────────────
SOURCE = "arco"                          # "arco" or "ifs"
INIT_TIME_STR = "2022-07-01T00:00:00Z"  # Myanmar monsoon season; within ARCO 1959-2023
# For IFS live run: SOURCE = "ifs"; INIT_TIME_STR = None

if SOURCE == "arco":
    from earth2studio.data import ARCO
    data_source = ARCO()
    INIT_TIME = datetime.fromisoformat(INIT_TIME_STR.replace("Z", "+00:00"))
    SOURCE_LABEL = "ARCO ERA5 reanalysis (Google Cloud, no credentials)"
    SOURCE_ATTR  = (
        "ERA5 via ARCO (Analysis-Ready Cloud-Optimized ERA5). "
        "Hosted on Google Cloud. ERA5 © ECMWF/Copernicus."
    )

elif SOURCE == "ifs":
    from earth2studio.data import IFS
    data_source = IFS()
    if INIT_TIME_STR:
        INIT_TIME = datetime.fromisoformat(INIT_TIME_STR.replace("Z", "+00:00"))
    else:
        # Auto-detect latest available IFS analysis (00Z or 12Z, ~6h lag)
        now = datetime.now(timezone.utc)
        for d in range(3):
            for h in [12, 0]:
                t = (now - timedelta(days=d)).replace(hour=h, minute=0, second=0, microsecond=0)
                if (now - t).total_seconds() >= 6 * 3600:
                    INIT_TIME = t
                    break
            else:
                continue
            break
    SOURCE_LABEL = "IFS HRES open data (ECMWF, no credentials)"
    SOURCE_ATTR  = (
        "IFS HRES analysis — ECMWF open data. "
        "CC BY 4.0. https://confluence.ecmwf.int/display/DAC/ECMWF+open+data"
    )
else:
    raise ValueError(f"Unknown SOURCE: {SOURCE}")

print(f"Source    : {SOURCE_LABEL}")
print(f"Init time : {INIT_TIME.isoformat()}")

## Section 6 — Load GraphCastSmall weights

Downloads checkpoint from `gs://dm_graphcast/graphcast` (~300 MB).  
**Do NOT call `model.to(bfloat16)`** — GraphCastSmall applies bfloat16 internally via JAX's `casting.Bfloat16Cast`.

In [ ]:
import time, torch, jax
from earth2studio.models.px import GraphCastSmall

def vram_snapshot(label):
    torch_alloc = torch.cuda.memory_allocated() / 1e9
    torch_res   = torch.cuda.memory_reserved()  / 1e9
    torch_free  = torch.cuda.mem_get_info()[0]  / 1e9
    jax_used = None
    try:
        devs = jax.devices("gpu")
        if devs:
            s = devs[0].memory_stats()
            if s:
                jax_used = s.get("bytes_in_use", 0) / 1e9
    except Exception:
        pass
    print(f"  [{label}]")
    print(f"    PyTorch: alloc={torch_alloc:.3f} GB  reserved={torch_res:.3f} GB  free={torch_free:.3f} GB")
    jax_str = f"{jax_used:.3f} GB" if jax_used is not None else "(opaque — JAX pool)"
    print(f"    JAX   : {jax_str}")

device = torch.device("cuda")
vram_snapshot("STAGE 1 — baseline before model load")

print("\nLoading GraphCastSmall...")
t0 = time.time()
try:
    package = GraphCastSmall.load_default_package()
    model   = GraphCastSmall.load_model(package)
    load_time = time.time() - t0
    print(f"✓ Model loaded in {load_time:.1f}s")
    print("  Checkpoint: GraphCast_small - ERA5 1979-2015 - resolution 1.0 - "
          "pressure levels 13 - mesh 2to5 - precipitation input and output.npz")
    print("  Backend: JAX + Haiku (bfloat16 internal via casting.Bfloat16Cast)")
except Exception as e:
    print(f"✗ Model load failed: {type(e).__name__}: {e}")
    raise

vram_snapshot("STAGE 2 — after model load")

## Section 7 — Stage 3: Smoke test (nsteps=1)

**This is the T4 hardware gate.**

If this cell raises `RuntimeError: CUDA out of memory`, **stop**. Do not modify the model or attempt memory workarounds. Capture the failure metrics printed below and report them.

In [ ]:
import earth2studio.run as e2run
from earth2studio.io import ZarrBackend

SMOKE_PASSED = False
smoke_time = None

print("[STAGE 3] Smoke test — nsteps=1 (single 6h inference step)")
print(f"  GPU: {GPU_NAME}  ({VRAM_TOTAL_GB:.1f} GB)")
print()

torch.cuda.reset_peak_memory_stats()
io_smoke = ZarrBackend()
t_smoke  = time.time()

try:
    with torch.inference_mode():
        io_smoke = e2run.deterministic(
            time=[INIT_TIME],
            nsteps=1,
            prognostic=model,
            data=data_source,
            io=io_smoke,
            device=device,
            verbose=True,
        )
    smoke_time = time.time() - t_smoke
    SMOKE_PASSED = True
    print(f"\n✓ SMOKE TEST PASSED in {smoke_time:.1f}s")
    vram_snapshot("STAGE 3 — after smoke test (nsteps=1)")

except RuntimeError as e:
    elapsed = time.time() - t_smoke
    print(f"\n✗ SMOKE TEST FAILED after {elapsed:.1f}s")
    print(f"  Error: {type(e).__name__}: {str(e)[:500]}")
    vram_snapshot("at OOM")
    peak_alloc = torch.cuda.max_memory_allocated() / 1e9
    peak_res   = torch.cuda.max_memory_reserved()  / 1e9
    print(f"\n  ── FAILURE REPORT ──")
    print(f"  GPU            : {GPU_NAME}")
    print(f"  VRAM total     : {VRAM_TOTAL_GB:.2f} GB")
    print(f"  Peak alloc     : {peak_alloc:.3f} GB (PyTorch tensors only)")
    print(f"  Peak reserved  : {peak_res:.3f} GB")
    print(f"  JAX pool       : separate (not reflected in PyTorch stats)")
    print(f"  Elapsed        : {elapsed:.1f}s")
    print()
    print("  GraphCastSmall is NOT viable on this GPU.")
    print("  Do NOT attempt workarounds (Constitution §XI).")
    print("  Alternatives: FourCastNet, Pangu-Weather, or a 16 GB-compatible model.")
    raise  # Stop notebook execution

assert SMOKE_PASSED, "Smoke test did not pass — cannot proceed"
print("\nProceeding to full 24h forecast.")

## Section 8 — Stage 4: Full 24h forecast (nsteps=4)

In [ ]:
assert SMOKE_PASSED, "Smoke test must pass before full run"

FULL_PASSED = False
inference_time = None

print(f"[STAGE 4] Full 24h forecast — nsteps={GC_N_STEPS}")
print(f"  {GC_N_STEPS} × {GC_STEP_HOURS}h steps → t+6h, t+12h, t+18h, t+24h")
print()

torch.cuda.reset_peak_memory_stats()
io_full  = ZarrBackend()
t_infer  = time.time()

try:
    with torch.inference_mode():
        io_full = e2run.deterministic(
            time=[INIT_TIME],
            nsteps=GC_N_STEPS,
            prognostic=model,
            data=data_source,
            io=io_full,
            device=device,
            verbose=True,
        )
    inference_time = time.time() - t_infer
    FULL_PASSED = True
    print(f"\n✓ FULL FORECAST PASSED in {inference_time:.1f}s ({inference_time/60:.1f} min)")
    vram_snapshot("STAGE 4 — after full forecast (nsteps=4)")
    PEAK_ALLOC_GB = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak PyTorch alloc: {PEAK_ALLOC_GB:.3f} GB")

except RuntimeError as e:
    elapsed = time.time() - t_infer
    print(f"\n✗ FULL FORECAST FAILED after {elapsed:.1f}s")
    print(f"  Note: 1-step smoke test passed but 4-step run OOM'd.")
    print(f"  This may indicate JAX JIT memory grows across auto-regressive steps.")
    print(f"  Error: {str(e)[:400]}")
    vram_snapshot("at OOM (full run)")
    raise

## Section 9 — Post-process: tp06 → mm/6h, Myanmar subset

**tp06 pipeline — no log/exp transform:**
```
GraphCastSmall zarr output: tp06 in physical metres
    ↓  × 1000
mm / 6h accumulation
    ↓  clamp ≥ 0  (physical constraint)
final float32 array [4, 21, 11]
    ↓  prepend t+0h zero frame
output [5, 21, 11]
```

In [ ]:
import numpy as np
import xarray as xr

assert FULL_PASSED

# Free model from GPU before post-processing
del model
torch.cuda.empty_cache()
print("Model released from GPU.")

ds = xr.open_zarr(io_full.store)
print(f"Zarr variables : {list(ds.data_vars)}")
print(f"Zarr dims      : {dict(ds.dims)}")

if "tp06" not in ds:
    raise ValueError(f"'tp06' not found. Available: {list(ds.data_vars)}")

tp06_raw = ds["tp06"]
print(f"tp06 raw dims  : {dict(tp06_raw.sizes)}")
print(f"tp06 lat range : {float(tp06_raw.lat.min()):.1f} to {float(tp06_raw.lat.max()):.1f}°")
print(f"tp06 lon range : {float(tp06_raw.lon.min()):.1f} to {float(tp06_raw.lon.max()):.1f}°")

# Myanmar subset — model lat is DESCENDING (90→-90)
tp06_myanmar = tp06_raw.sel(
    lat=slice(MYANMAR_LAT_MAX, MYANMAR_LAT_MIN),
    lon=slice(MYANMAR_LON_MIN, MYANMAR_LON_MAX),
).squeeze()

# Ensure ascending lat (9°N at index 0) for binary artifact
if tp06_myanmar.coords["lat"].values[0] > tp06_myanmar.coords["lat"].values[-1]:
    tp06_myanmar = tp06_myanmar.isel(lat=slice(None, None, -1))

lats_out = tp06_myanmar.coords["lat"].values.astype(np.float64)
lons_out = tp06_myanmar.coords["lon"].values.astype(np.float64)
n_lat_out, n_lon_out = len(lats_out), len(lons_out)

print(f"\nMyanmar subset : {n_lat_out} lat × {n_lon_out} lon (expected 21 × 11)")
print(f"Lat range      : {lats_out[0]:.1f}°N to {lats_out[-1]:.1f}°N")
print(f"Lon range      : {lons_out[0]:.1f}°E to {lons_out[-1]:.1f}°E")

# tp06: physical metres → mm/6h (NO exp/log transform)
tp06_m  = tp06_myanmar.values.astype(np.float32)           # metres
tp06_mm = np.maximum(tp06_m * 1000.0, 0.0).astype(np.float32)  # mm/6h

print(f"\ntp06 metres    : [{tp06_m.min():.6f}, {tp06_m.max():.6f}]")
print(f"tp06 mm/6h     : [{tp06_mm.min():.4f}, {tp06_mm.max():.4f}]")
print(f"tp06 shape     : {tp06_mm.shape}  dtype: {tp06_mm.dtype}")

# Prepend synthetic t+0h zero frame
tp06_full = np.concatenate(
    [np.zeros((1, n_lat_out, n_lon_out), dtype=np.float32), tp06_mm],
    axis=0,
)
print(f"\nFinal array    : {tp06_full.shape}  (expected [5, 21, 11])")
print(f"Binary size    : {tp06_full.size * 4} bytes  (expected 4620)")

## Section 10 — Precipitation statistics

In [ ]:
print("=" * 60)
print("PRECIPITATION SANITY CHECK (tp06)")
print("=" * 60)

n_nan   = int(np.sum(np.isnan(tp06_full)))
n_inf   = int(np.sum(np.isinf(tp06_full)))
n_neg   = int(np.sum(tp06_full < 0.0))
n_zero  = int(np.sum(tp06_full == 0.0))
n_total = tp06_full.size
zero_pct = 100.0 * n_zero / n_total

valid   = tp06_full[np.isfinite(tp06_full)]
vmin    = float(valid.min())
vmax    = float(valid.max())
vmedian = float(np.median(valid))
p95     = float(np.percentile(valid, 95))
p99     = float(np.percentile(valid, 99))

print(f"  Variable            : tp06")
print(f"  Unit                : mm / 6h accumulation")
print(f"  Transform           : metres × 1000 (NO exp/log)")
print(f"  Shape               : {tp06_full.shape}")
print(f"  Min                 : {vmin:.4f} mm/6h")
print(f"  Median              : {vmedian:.4f} mm/6h")
print(f"  P95                 : {p95:.4f} mm/6h")
print(f"  P99                 : {p99:.4f} mm/6h")
print(f"  Max                 : {vmax:.4f} mm/6h")
print(f"  Zero-rain fraction  : {zero_pct:.1f}%  (includes synthetic t+0h)")
print(f"  NaN                 : {n_nan}")
print(f"  Inf                 : {n_inf}")
print(f"  Negative            : {n_neg}")
print()
print("  Per-frame max (mm/6h):")
for i, fmax in enumerate(tp06_full.max(axis=(1, 2))):
    note = " ← synthetic t+0h zero" if i == 0 else ""
    print(f"    t+{i*6:2d}h frame {i}: {fmax:.4f}{note}")

failures = []
if n_nan > 0:  failures.append(f"NaN: {n_nan}")
if n_inf > 0:  failures.append(f"Inf: {n_inf}")
if n_neg > 0:  failures.append(f"Negative after clamping: {n_neg}")
if vmax > SANITY_MAX_MM: failures.append(f"Max {vmax:.1f} > {SANITY_MAX_MM:.0f} mm/6h threshold")

print()
PRECIP_PASSED = not bool(failures)
if failures:
    print("  Status : FAIL")
    for f in failures:
        print(f"    ✗ {f}")
    print("  Data is written as-is — never silently modified.")
else:
    print("  Status : PASS ✓")
print("=" * 60)

## Section 11 — Write artifacts

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timedelta, timezone
import earth2studio

OUTPUT_DIR = Path("data/forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

times_utc = [
    (INIT_TIME + timedelta(hours=i * GC_STEP_HOURS)).strftime("%Y-%m-%dT%H:%M:%SZ")
    for i in range(GC_N_FRAMES)
]

# ── precipitation.bin ─────────────────────────────────────────────────────────
precip_path = OUTPUT_DIR / "precipitation.bin"
precip_path.write_bytes(tp06_full.astype("<f4").tobytes())
actual_bytes = precip_path.stat().st_size
expected_bytes = GC_N_FRAMES * n_lat_out * n_lon_out * 4
assert actual_bytes == expected_bytes, f"Size mismatch: {actual_bytes} != {expected_bytes}"
print(f"precipitation.bin: {actual_bytes} bytes ✓")

# ── forecast.json ─────────────────────────────────────────────────────────────
meta = {
    "schema_version": "2.0",
    "model": "GraphCastSmall",
    "model_version": "1.0",
    "model_checkpoint": (
        "GraphCast_small - ERA5 1979-2015 - resolution 1.0 - "
        "pressure levels 13 - mesh 2to5 - precipitation input and output.npz"
    ),
    "model_source": "gs://dm_graphcast/graphcast",
    "model_attribution": (
        "GraphCast by DeepMind/Google. Lam et al. (2023), Science. "
        "https://arxiv.org/abs/2212.12794. Earth2Studio wrapper by NVIDIA."
    ),
    "initialization_source": SOURCE.upper(),
    "initialization_time": times_utc[0],
    "forecast_generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "forecast_horizon_hours": GC_N_STEPS * GC_STEP_HOURS,
    "native_timestep_hours": GC_STEP_HOURS,
    "n_times": GC_N_FRAMES,
    "spatial_resolution_deg": 1.0,
    "display_resolution_deg": None,
    "region": "Myanmar",
    "bbox": {
        "lat_min": float(lats_out.min()), "lat_max": float(lats_out.max()),
        "lon_min": float(lons_out.min()), "lon_max": float(lons_out.max()),
    },
    "grid": {"n_lat": n_lat_out, "n_lon": n_lon_out},
    "lat": lats_out.tolist(),
    "lon": lons_out.tolist(),
    "times_utc": times_utc,
    "variables": {
        "precipitation": {
            "display_name": "Precipitation",
            "units": "mm / 6h",
            "source_variable": "tp06",
            "temporal_resolution": "6-hourly",
            "temporal_semantics": (
                "Total precipitation accumulated over the 6-hour forecast "
                "period ending at the displayed valid time."
            ),
            "temporal_disclosure": (
                "Precipitation values represent total rainfall accumulated "
                "during the 6-hour forecast period ending at the displayed time. "
                "These are not instantaneous rainfall rates."
            ),
            "transformation_provenance": {
                "source_variable": "tp06",
                "source_unit": "metres",
                "conversion": "metres × 1000",
                "output_unit": "mm",
                "accumulation_period_hours": GC_STEP_HOURS,
                "log_transform_applied": False,
                "exp_transform_applied": False,
                "pipeline": (
                    "GraphCastSmall native tp06 (physical metres, no log transform) "
                    "→ × 1000 → mm/6h → clamp ≥ 0"
                ),
            },
            "t0_note": (
                "t+0h frame (index 0) is synthetic: precipitation = 0.0. "
                "Represents analysis state; no forecast accumulation has occurred."
            ),
            "native_output": True,
            "file": "precipitation.bin",
            "fill_value": None,
        }
    },
    "data_source_attribution": SOURCE_ATTR,
    "earth2studio_version": earth2studio.__version__,
    "inference_config": {
        "device": GPU_NAME,
        "peak_vram_gb": round(PEAK_ALLOC_GB, 3),
        "vram_total_gb": round(VRAM_TOTAL_GB, 1),
        "jax_env": {
            "XLA_PYTHON_CLIENT_PREALLOCATE": os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"),
            "XLA_PYTHON_CLIENT_MEM_FRACTION": os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"),
        },
        "inference_time_seconds": round(inference_time),
        "model_load_time_seconds": round(load_time),
    },
    "is_demo": False,
}

json_path = OUTPUT_DIR / "forecast.json"
with open(json_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"forecast.json   : {json_path.stat().st_size} bytes ✓")
print(f"\nArtifacts in {OUTPUT_DIR.resolve()}:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}: {p.stat().st_size:,} bytes")

## Section 12 — Validate artifacts

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "scripts/validate_forecast.py", "--data-dir", str(OUTPUT_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"Validation failed (exit {result.returncode})")
VALIDATE_PASSED = True
print("✓ All validation checks passed")

## Section 13 — Full provenance report

In [ ]:
print("=" * 62)
print("GRAPHCASTSMALL T4 VALIDATION REPORT")
print("=" * 62)
print()
print("MODEL")
print(f"  Name              : GraphCastSmall")
print(f"  Checkpoint        : GraphCast_small - ERA5 1979-2015 - resolution 1.0 - ...")
print(f"  Earth2Studio      : {earth2studio.__version__}")
print(f"  Native resolution : 1.0° (181 × 360 global)")
print()
print("INITIALIZATION")
print(f"  Source            : {SOURCE.upper()}")
print(f"  Init time         : {INIT_TIME.isoformat()}")
print(f"  Input timesteps   : t-6h and t+0h (two-timestep requirement, auto by Earth2Studio)")
print()
print("FORECAST")
print(f"  Horizon           : 24h")
print(f"  Step              : 6h")
print(f"  Frames            : 5 (t+0h .. t+24h)")
print(f"  Times             : {', '.join(times_utc)}")
print()
print("HARDWARE")
print(f"  GPU               : {GPU_NAME}")
print(f"  VRAM total        : {VRAM_TOTAL_GB:.1f} GB")
print(f"  Peak alloc (torch): {PEAK_ALLOC_GB:.3f} GB")
print(f"  Model load        : {load_time:.1f}s")
print(f"  Inference (4 step): {inference_time:.1f}s = {inference_time/60:.1f} min")
print()
print("PRECIPITATION")
print(f"  Variable          : tp06")
print(f"  Unit              : mm / 6h")
print(f"  Accumulation      : 6-hour total")
print(f"  Min               : {vmin:.4f}")
print(f"  Median            : {vmedian:.4f}")
print(f"  P95               : {p95:.4f}")
print(f"  P99               : {p99:.4f}")
print(f"  Max               : {vmax:.4f}")
print(f"  NaN               : {n_nan}")
print(f"  Inf               : {n_inf}")
print(f"  Negative          : {n_neg}")
print(f"  Zero-rain frac    : {zero_pct:.1f}%")
print()
print("OUTPUT")
print(f"  Shape             : {tp06_full.shape}")
print(f"  Dtype             : {tp06_full.dtype}")
print(f"  precipitation.bin : {actual_bytes} bytes")
print(f"  forecast.json     : {json_path.stat().st_size} bytes")
print()
print("STATUS")
smoke_s = '✓ PASS' if SMOKE_PASSED else '✗ FAIL'
full_s  = '✓ PASS' if FULL_PASSED  else '✗ FAIL'
prec_s  = '✓ PASS' if PRECIP_PASSED else '✗ FAIL'
val_s   = '✓ PASS' if VALIDATE_PASSED else '✗ FAIL'
print(f"  T4 inference (smoke test) : {smoke_s}")
print(f"  6h smoke test             : {smoke_s}")
print(f"  24h forecast              : {full_s}")
print(f"  Precipitation QC          : {prec_s}")
print(f"  Artifact validation       : {val_s}")
print("=" * 62)

## Section 14 — Git commit and push to GitHub

Commits `data/forecast/` and pushes to `main`, triggering GitHub Actions → GitHub Pages deployment.

**Requires**: `GITHUB_TOKEN` Colab secret (Contents: Read + Write on the repository).

In [ ]:
import subprocess, json as _json
from google.colab import userdata

REPO_OWNER = "JiayanLim"
REPO_NAME  = "myanmar-weather-forecast"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    print("⚠ No GITHUB_TOKEN — skipping push. Add token to Colab secrets and re-run.")
else:
    # Configure git
    subprocess.run(["git", "config", "user.email", "colab@graphcastsmall"])
    subprocess.run(["git", "config", "user.name",  "GraphCastSmall Colab"])
    subprocess.run([
        "git", "remote", "set-url", "origin",
        f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    ], capture_output=True)

    # Pull first to avoid non-fast-forward push
    pr = subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
    print(pr.stdout.strip() or "Already up to date.")

    # Stage artifacts
    subprocess.run(["git", "add", "data/forecast/precipitation.bin", "data/forecast/forecast.json"])

    meta_r = _json.loads(open("data/forecast/forecast.json").read())
    init_t = meta_r["initialization_time"]
    gpu_r  = meta_r["inference_config"]["device"]
    inf_s  = meta_r["inference_config"]["inference_time_seconds"]

    commit_msg = (
        f"feat: GraphCastSmall 24h Myanmar forecast {init_t}\n\n"
        f"Model: GraphCastSmall 1.0° / 6h steps / tp06 / schema v2.0\n"
        f"Source: {SOURCE.upper()}\n"
        f"GPU: {gpu_r}\n"
        f"Inference: {inf_s}s\n"
        f"Shape: [5, 21, 11] float32 / {actual_bytes} bytes\n"
    )
    cr = subprocess.run(["git", "commit", "-m", commit_msg], capture_output=True, text=True)
    print(cr.stdout.strip() or cr.stderr.strip())

    push = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)
    if push.returncode == 0:
        print("\n✓ Pushed to main. GitHub Actions will deploy to GitHub Pages.")
        print(f"  Actions: https://github.com/{REPO_OWNER}/{REPO_NAME}/actions")
        print(f"  Pages  : https://jiayanlim.github.io/myanmar-weather-forecast/")
    else:
        print("✗ Push failed:", push.stderr)